In [1]:

# D1 - Kiểm tra môi trường làm việc
import pandas as pd
import numpy as np
import sklearn

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score, recall_score

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Isolation Forest: OK")
print("LOF: OK")
print("Environment: READY")


Pandas: 3.0.2
NumPy: 2.5.4
Scikit-learn: 1.8.0
Isolation Forest: OK
LOF: OK
Environment: READY


In [2]:

from pathlib import Path
import pandas as pd

# Tìm thư mục gốc của dự án
root = Path.cwd()

if not (root / "data").exists():
    root = root.parent

# Đường dẫn dữ liệu
data_path = root / "data" / "raw" / "creditcard.csv"

if not data_path.exists():
    raise FileNotFoundError(
        f"Không tìm thấy dữ liệu: {data_path}"
    )

# Đọc dữ liệu
df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Số dòng:", df.shape[0])
print("Số cột:", df.shape[1])
print("\nDanh sách cột:")
print(df.columns.tolist())

display(df.head())


Dataset loaded successfully!
Số dòng: 284807
Số cột: 31

Danh sách cột:
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:

# D1 - Kiểm tra dữ liệu trước khi huấn luyện

# 1. Kiểm tra kích thước
print("Kích thước dataset:", df.shape)

# 2. Kiểm tra giá trị thiếu
print("\nSố giá trị thiếu:")
print(df.isnull().sum())

# 3. Kiểm tra bản ghi trùng lặp
print("\nSố dòng trùng lặp:", df.duplicated().sum())

# 4. Kiểm tra phân bố nhãn
print("\nPhân bố Class:")
print(df["Class"].value_counts())

print("\nTỷ lệ phần trăm:")
print(df["Class"].value_counts(normalize=True) * 100)

# 5. Kiểm tra các cột đặc trưng
print("\nDanh sách feature:")
features = df.drop(columns=["Class"])

print(features.columns.tolist())
print("Số feature:", features.shape[1])


Kích thước dataset: (284807, 31)

Số giá trị thiếu:
Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64

Số dòng trùng lặp: 1081

Phân bố Class:
Class
0    284315
1       492
Name: count, dtype: int64

Tỷ lệ phần trăm:
Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64

Danh sách feature:
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']
Số feature: 30


In [4]:

# D1 - Tổng hợp chất lượng dữ liệu đầu vào

print("=== DATA QUALITY CHECK ===")

print("1. Tổng số giá trị thiếu:", df.isna().sum().sum())

print("2. Tổng số dòng trùng lặp:", df.duplicated().sum())

print("\n3. Phân bố nhãn:")
print(df["Class"].value_counts().sort_index())

print("\n4. Tỷ lệ gian lận (%):")
fraud_rate = df["Class"].mean() * 100
print(f"{fraud_rate:.4f}%")

print("\n5. Kiểm tra nhãn hợp lệ:")
print(sorted(df["Class"].dropna().unique().tolist()))


=== DATA QUALITY CHECK ===
1. Tổng số giá trị thiếu: 0
2. Tổng số dòng trùng lặp: 1081

3. Phân bố nhãn:
Class
0    284315
1       492
Name: count, dtype: int64

4. Tỷ lệ gian lận (%):
0.1727%

5. Kiểm tra nhãn hợp lệ:
[0, 1]


In [5]:

# D2 - Xác định đặc trưng và nhãn đánh giá

feature_cols = [f"V{i}" for i in range(1, 29)]
feature_cols += ["Time", "Amount"]

X_raw = df[feature_cols].copy()
y = df["Class"].copy()

print("=== MODEL INPUT CHECK ===")
print("Kích thước X:", X_raw.shape)
print("Kích thước y:", y.shape)
print("Có Class trong X?", "Class" in X_raw.columns)

print("\nKiểu dữ liệu:")
print(X_raw.dtypes.value_counts())

print("\nSố giá trị vô hạn:",
      np.isinf(X_raw.to_numpy()).sum())

print("\nThống kê Time và Amount:")
display(X_raw[["Time", "Amount"]].describe())


=== MODEL INPUT CHECK ===
Kích thước X: (284807, 30)
Kích thước y: (284807,)
Có Class trong X? False

Kiểu dữ liệu:
float64    30
Name: count, dtype: int64

Số giá trị vô hạn: 0

Thống kê Time và Amount:


,Time,Amount
count,284807.000000,284807.000000
mean,94813.859575,88.349619
std,47488.145955,250.120109
min,0.000000,0.000000
25%,54201.500000,5.600000
50%,84692.000000,22.000000
75%,139320.500000,77.165000
max,172792.000000,25691.160000


In [6]:

from sklearn.model_selection import train_test_split

# Giữ nguyên dữ liệu gốc, chưa chuẩn hóa hoặc xóa trùng
indices = np.arange(len(df))

# Chia 70% train, 30% còn lại
idx_train, idx_temp = train_test_split(
    indices,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Chia 30% thành validation và test
idx_val, idx_test = train_test_split(
    idx_temp,
    test_size=0.50,
    stratify=y.iloc[idx_temp],
    random_state=42
)

print("=== DATA SPLIT ===")

for name, idx in [
    ("Train", idx_train),
    ("Validation", idx_val),
    ("Test", idx_test)
]:
    labels = y.iloc[idx]
    print(
        f"{name}: {len(idx):,} giao dịch | "
        f"Fraud: {labels.sum()} | "
        f"Fraud rate: {labels.mean():.4%}"
    )


=== DATA SPLIT ===
Train: 199,364 giao dịch | Fraud: 344 | Fraud rate: 0.1725%
Validation: 42,721 giao dịch | Fraud: 74 | Fraud rate: 0.1732%
Test: 42,722 giao dịch | Fraud: 74 | Fraud rate: 0.1732%


In [7]:

# D3 - Kiểm tra trùng lặp giữa Train / Validation / Test

train_df = df.iloc[idx_train].copy()
val_df = df.iloc[idx_val].copy()
test_df = df.iloc[idx_test].copy()

# Kiểm tra trên toàn bộ 31 cột, bao gồm cả nhãn Class
def check_overlap(source, target):
    source_unique = source.drop_duplicates()
    matched = target.merge(
        source_unique,
        how="inner",
        on=list(df.columns)
    )
    return len(matched)

print("=== CROSS-SPLIT DUPLICATE CHECK ===")

print("Train - Validation:",
      check_overlap(train_df, val_df))

print("Train - Test:",
      check_overlap(train_df, test_df))

print("Validation - Test:",
      check_overlap(val_df, test_df))


=== CROSS-SPLIT DUPLICATE CHECK ===
Train - Validation: 207
Train - Test: 249
Validation - Test: 61


In [8]:

# Kiểm tra bản ghi trùng theo nhãn Class

duplicate_mask = df.duplicated(keep="first")

print("=== DUPLICATE ANALYSIS ===")

print("Tổng số dòng trùng:",
      duplicate_mask.sum())

print("\nSố dòng trùng theo Class:")
print(
    df.loc[duplicate_mask, "Class"]
      .value_counts()
      .sort_index()
)

print("\nSố nhóm bản ghi có nhiều hơn một dòng:")
group_sizes = df.groupby(
    list(df.columns),
    dropna=False
).size()

print((group_sizes > 1).sum())

print("\nSố lần lặp lớn nhất của một bản ghi:")
print(group_sizes.max())


=== DUPLICATE ANALYSIS ===
Tổng số dòng trùng: 1081

Số dòng trùng theo Class:
Class
0    1062
1      19
Name: count, dtype: int64

Số nhóm bản ghi có nhiều hơn một dòng:
773

Số lần lặp lớn nhất của một bản ghi:
18


In [9]:

# D5 - Isolation Forest: chạy thử thuật toán
# Chưa phải kết quả chính thức của nhóm

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Chỉ dùng các cột V1-V28 để thử nghiệm ban đầu
# Time và Amount sẽ được xử lý theo quyết định của Cường
demo_features = [f"V{i}" for i in range(1, 29)]

X_train_demo = df.iloc[idx_train][demo_features]

# Chuẩn hóa chỉ bằng tập train
iso_demo = Pipeline([
    ("scaler", StandardScaler()),
    ("model", IsolationForest(
        n_estimators=100,
        contamination="auto",
        random_state=42,
        n_jobs=-1
    ))
])

iso_demo.fit(X_train_demo)

print("Isolation Forest trained successfully!")
print("Số dòng huấn luyện:", len(X_train_demo))
print("Số feature:", len(demo_features))


Isolation Forest trained successfully!
Số dòng huấn luyện: 199364
Số feature: 28


In [10]:

# D8 - Tạo điểm bất thường trên validation (thử nghiệm)

X_val_demo = df.iloc[idx_val][demo_features]

# decision_function: điểm càng thấp càng bất thường
# Đảo dấu để điểm càng cao càng bất thường
val_scores = -iso_demo.decision_function(X_val_demo)

val_results = pd.DataFrame({
    "original_index": idx_val,
    "anomaly_score": val_scores,
    "is_fraud": y.iloc[idx_val].to_numpy()
})

val_results["rank"] = (
    val_results["anomaly_score"]
    .rank(ascending=False, method="first")
    .astype(int)
)

display(
    val_results.sort_values("rank").head(10)
)


,original_index,anomaly_score,is_fraud,rank
38001,153823,0.239705,1,1
26120,43204,0.233140,1,2
34929,46841,0.222831,0,3
1925,173353,0.221901,0,4
39994,138706,0.214511,0,5
30209,219257,0.213916,0,6
16090,50287,0.213232,0,7
22663,145022,0.207388,0,8
36180,155741,0.207182,0,9
3700,176335,0.206110,0,10


In [11]:

# D9 - Đánh giá Isolation Forest trên Validation
# Kết quả thử nghiệm, chưa phải kết quả chính thức

from sklearn.metrics import average_precision_score
import numpy as np

# Sắp xếp giao dịch theo điểm bất thường giảm dần
ranked = val_results.sort_values(
    "anomaly_score",
    ascending=False
).copy()

# Lấy 100 giao dịch bất thường nhất
k = min(100, len(ranked))
top_k = ranked.head(k)

# Số giao dịch gian lận tìm thấy
fraud_detected = int(top_k["is_fraud"].sum())

# Tổng giao dịch gian lận trong validation
total_fraud = int(ranked["is_fraud"].sum())

# Tính các chỉ số
precision_at_100 = fraud_detected / k

recall_at_100 = (
    fraud_detected / total_fraud
    if total_fraud > 0 else 0
)

pr_auc = average_precision_score(
    ranked["is_fraud"],
    ranked["anomaly_score"]
)

print("=== ISOLATION FOREST EVALUATION ===")
print("Total transactions:", len(ranked))
print("Total fraud:", total_fraud)
print("Fraud detected in Top 100:", fraud_detected)
print(f"Precision@100: {precision_at_100:.4%}")
print(f"Recall@100: {recall_at_100:.4%}")
print(f"Average Precision (AP): {pr_auc:.4f}")


=== ISOLATION FOREST EVALUATION ===
Total transactions: 42721
Total fraud: 74
Fraud detected in Top 100: 24
Precision@100: 24.0000%
Recall@100: 32.4324%
Average Precision (AP): 0.1832


In [12]:

# D6 - Local Outlier Factor (LOF)
# Thử nghiệm kỹ thuật, chưa phải mô hình chính thức

from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np

# Chọn mẫu huấn luyện chỉ từ tập train
# Class chỉ dùng để tạo tập tham chiếu bình thường
normal_train_idx = idx_train[
    y.iloc[idx_train].to_numpy() == 0
]

rng = np.random.default_rng(42)

sample_idx = rng.choice(
    normal_train_idx,
    size=min(20000, len(normal_train_idx)),
    replace=False
)

X_train_lof = df.iloc[sample_idx][demo_features]
X_val_lof = df.iloc[idx_val][demo_features]

lof_demo = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LocalOutlierFactor(
        n_neighbors=20,
        novelty=True,
        n_jobs=-1
    ))
])

lof_demo.fit(X_train_lof)

print("LOF trained successfully!")
print("Training samples:", len(X_train_lof))
print("Features:", X_train_lof.shape[1])


LOF trained successfully!
Training samples: 20000
Features: 28


In [13]:

# D8-D9: Đánh giá LOF trên Validation
# Chỉ là kết quả thử nghiệm

from sklearn.metrics import average_precision_score
import pandas as pd
import numpy as np

# Điểm càng cao = càng bất thường
lof_scores = -lof_demo.decision_function(X_val_lof)

lof_results = pd.DataFrame({
    "original_index": idx_val,
    "anomaly_score": lof_scores,
    "is_fraud": y.iloc[idx_val].to_numpy()
})

# Sắp xếp từ bất thường nhất đến ít bất thường nhất
lof_ranked = lof_results.sort_values(
    "anomaly_score", ascending=False
)

k = 100
lof_top100 = lof_ranked.head(k)

fraud_detected_lof = int(lof_top100["is_fraud"].sum())
total_fraud_lof = int(lof_ranked["is_fraud"].sum())

precision_lof = fraud_detected_lof / k
recall_lof = fraud_detected_lof / total_fraud_lof
ap_lof = average_precision_score(
    lof_results["is_fraud"],
    lof_results["anomaly_score"]
)

print("=== LOF EVALUATION ===")
print("Total transactions:", len(lof_results))
print("Total fraud:", total_fraud_lof)
print("Fraud detected in Top 100:", fraud_detected_lof)
print(f"Precision@100: {precision_lof:.4%}")
print(f"Recall@100: {recall_lof:.4%}")
print(f"Average Precision (AP): {ap_lof:.4f}")


=== LOF EVALUATION ===
Total transactions: 42721
Total fraud: 74
Fraud detected in Top 100: 49
Precision@100: 49.0000%
Recall@100: 66.2162%
Average Precision (AP): 0.4211


In [14]:

# D6 - LOF không giám sát: thí nghiệm thử
# Không sử dụng Class để chọn dữ liệu huấn luyện

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
import numpy as np

rng = np.random.default_rng(42)

# Lấy ngẫu nhiên 20.000 giao dịch từ tập train
sample_idx_unsupervised = rng.choice(
    idx_train,
    size=min(20000, len(idx_train)),
    replace=False
)

X_train_lof_unsup = df.iloc[
    sample_idx_unsupervised
][demo_features]

lof_unsup = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LocalOutlierFactor(
        n_neighbors=20,
        novelty=True,
        n_jobs=-1
    ))
])

lof_unsup.fit(X_train_lof_unsup)

print("Unsupervised LOF trained successfully!")
print("Training samples:", len(X_train_lof_unsup))
print("Features:", X_train_lof_unsup.shape[1])


Unsupervised LOF trained successfully!
Training samples: 20000
Features: 28


In [15]:

# Đánh giá LOF không giám sát trên Validation

from sklearn.metrics import average_precision_score
import pandas as pd

unsup_scores = -lof_unsup.decision_function(
    X_val_lof
)

unsup_results = pd.DataFrame({
    "anomaly_score": unsup_scores,
    "is_fraud": y.iloc[idx_val].to_numpy()
})

unsup_ranked = unsup_results.sort_values(
    "anomaly_score",
    ascending=False
)

detected = int(
    unsup_ranked.head(100)["is_fraud"].sum()
)

total_fraud = int(unsup_results["is_fraud"].sum())

precision_unsup = detected / 100
recall_unsup = detected / total_fraud

ap_unsup = average_precision_score(
    unsup_results["is_fraud"],
    unsup_results["anomaly_score"]
)

print("=== UNSUPERVISED LOF EVALUATION ===")
print("Fraud detected:", detected)
print(f"Precision@100: {precision_unsup:.4%}")
print(f"Recall@100: {recall_unsup:.4%}")
print(f"Average Precision: {ap_unsup:.4f}")


=== UNSUPERVISED LOF EVALUATION ===
Fraud detected: 3
Precision@100: 3.0000%
Recall@100: 4.0541%
Average Precision: 0.0134


In [16]:

# D7 - Phân tích anomaly score của LOF không giám sát

import pandas as pd
import numpy as np

scores_check = pd.DataFrame({
    "score": unsup_scores,
    "Class": y.iloc[idx_val].to_numpy()
})

print("=== LOF SCORE ANALYSIS ===")

print("\nThống kê điểm theo Class:")
print(
    scores_check.groupby("Class")["score"]
    .describe()
)

print("\nĐiểm trung bình:")
print(
    scores_check.groupby("Class")["score"]
    .mean()
)

print("\nTop 10 điểm bất thường cao nhất:")
display(
    scores_check.sort_values(
        "score",
        ascending=False
    ).head(10)
)


=== LOF SCORE ANALYSIS ===

Thống kê điểm theo Class:
         count      mean       std       min       25%       50%       75%  \
Class                                                                        
0      42647.0 -0.327430  0.221717 -0.567844 -0.471786 -0.391024 -0.255766   
1         74.0 -0.032547  0.551057 -0.483781 -0.407836 -0.217837  0.128091   

            max  
Class            
0      4.369824  
1      2.285352  

Điểm trung bình:
Class
0   -0.327430
1   -0.032547
Name: score, dtype: float64

Top 10 điểm bất thường cao nhất:


,score,Class
34929,4.369824,0
10255,4.003521,0
39994,3.496498,0
39529,3.271396,0
3957,2.765127,0
22663,2.705979,0
41499,2.485778,0
21255,2.458571,0
7050,2.285352,1
8004,2.285352,1


In [17]:

# D7 - Phân tích Precision@K và Recall@K
# Thí nghiệm trên Validation

import pandas as pd

ranked_lof = scores_check.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

total_fraud = int(ranked_lof["Class"].sum())

results_k = []

for k in [10, 20, 50, 100, 200, 500, 1000]:
    top_k = ranked_lof.head(k)

    fraud_found = int(top_k["Class"].sum())

    results_k.append({
        "Top K": k,
        "Fraud Detected": fraud_found,
        "Precision@K": fraud_found / k,
        "Recall@K": fraud_found / total_fraud
    })

topk_df = pd.DataFrame(results_k)

display(
    topk_df.style.format({
        "Precision@K": "{:.2%}",
        "Recall@K": "{:.2%}"
    })
)


,Top K,Fraud Detected,Precision@K,Recall@K
0,10,2,20.00%,2.70%
1,20,2,10.00%,2.70%
2,50,2,4.00%,2.70%
3,100,3,3.00%,4.05%
4,200,5,2.50%,6.76%
5,500,10,2.00%,13.51%
6,1000,17,1.70%,22.97%


In [18]:

# D7 - Tinh chỉnh n_neighbors của LOF
# Thí nghiệm trên Validation, chưa phải kết quả cuối cùng

from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
import pandas as pd

# Chuẩn hóa bằng tập train đã chọn
scaler_lof = StandardScaler()
X_train_scaled = scaler_lof.fit_transform(
    X_train_lof_unsup
)

X_val_scaled = scaler_lof.transform(X_val_lof)

y_val = y.iloc[idx_val].to_numpy()

tuning_results = []

for n in [10, 20, 50, 100]:
    model = LocalOutlierFactor(
        n_neighbors=n,
        novelty=True,
        n_jobs=-1
    )

    model.fit(X_train_scaled)

    scores = -model.decision_function(X_val_scaled)

    top100_idx = scores.argsort()[::-1][:100]
    detected = int(y_val[top100_idx].sum())

    tuning_results.append({
        "n_neighbors": n,
        "Fraud Top 100": detected,
        "Precision@100": detected / 100,
        "Recall@100": detected / y_val.sum(),
        "AP": average_precision_score(
            y_val, scores
        )
    })

tuning_df = pd.DataFrame(tuning_results)

display(
    tuning_df.style.format({
        "Precision@100": "{:.2%}",
        "Recall@100": "{:.2%}",
        "AP": "{:.4f}"
    })
)


,n_neighbors,Fraud Top 100,Precision@100,Recall@100,AP
0,10,3,3.00%,4.05%,0.0057
1,20,3,3.00%,4.05%,0.0134
2,50,31,31.00%,41.89%,0.1910
3,100,39,39.00%,52.70%,0.3266


In [19]:

# D7 - Lưu kết quả tinh chỉnh LOF

from pathlib import Path
import pandas as pd

output_dir = root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Lưu toàn bộ kết quả, không chỉ cấu hình tốt nhất
tuning_df.to_csv(
    output_dir / "lof_tuning_validation_draft.csv",
    index=False
)

# Lấy cấu hình tốt nhất theo Precision@100
best_row = tuning_df.sort_values(
    ["Precision@100", "AP"],
    ascending=False
).iloc[0]

print("=== BEST LOF CONFIGURATION (VALIDATION) ===")
print("n_neighbors:", int(best_row["n_neighbors"]))
print("Precision@100:", f'{best_row["Precision@100"]:.2%}')
print("Recall@100:", f'{best_row["Recall@100"]:.2%}')
print("Average Precision:", f'{best_row["AP"]:.4f}')

print("\nĐã lưu kết quả thử nghiệm.")


=== BEST LOF CONFIGURATION (VALIDATION) ===
n_neighbors: 100
Precision@100: 39.00%
Recall@100: 52.70%
Average Precision: 0.3266

Đã lưu kết quả thử nghiệm.
